# 01 — Architecture tour of `skillopt`

SkillOpt's optimizer is a batch trainer (`scripts/train.py`), but the *parts* are interesting: a Trainer in `engine/`, reflection in `gradient/`, edit operators in `optimizer/`, schedules in `scheduler/`, and a validation gate in `evaluation/`. This notebook walks the package live with `inspect.getsource`.

Zero LLM calls. Pure code archaeology.

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/02-skillopt.md`.

In [1]:
import inspect
import os

import skillopt

root = os.path.dirname(skillopt.__file__)
print("skillopt package:", root)
print()
# Tree of the source tree, two levels deep
for r, dirs, files in os.walk(root):
    if "__pycache__" in r or ".egg-info" in r:
        continue
    rel = os.path.relpath(r, root)
    depth = 0 if rel == "." else rel.count(os.sep) + 1
    indent = "  " * depth
    print(f"{indent}{os.path.basename(r) or '(skillopt)'}/")
    for f in sorted(files):
        if f.endswith(".py") and f != "__init__.py":
            print(f"{indent}  {f}")
    if depth >= 2:
        dirs.clear()

skillopt package: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt

skillopt/
  config.py
  types.py
  gradient/
    aggregate.py
    reflect.py
  optimizer/
    clip.py
    lr_autonomous.py
    meta_skill.py
    rewrite.py
    scheduler.py
    select.py
    skill.py
    slow_update.py
    update_modes.py
  datasets/
    base.py
  scheduler/
  utils/
    json_utils.py
    scoring.py
  model/
    azure_openai.py
    backend_config.py
    claude_backend.py
    codex_backend.py
    codex_harness.py
    common.py
    qwen_backend.py
    router.py
  prompts/
  evaluation/
    gate.py
  envs/
    base.py
    alfworld/
      adapter.py
      dataloader.py
      reflect.py
      rollout.py
    livemathematicianbench/
      adapter.py
      dataloader.py
      evaluator.py
      reflect.py
      rollout.py
    _template/
      env_template.py
      loader_template.py
    spreadsheetbench/
      adapter.py
      codegen_agent.py
      dataloader.py
      evaluator.py
      executor.py
      reac

## 1. The Trainer — `engine/trainer.py`

The Trainer is the main loop. It owns: the skill, the dataset, the optimizer, the scheduler, and the gate.

In [4]:
from skillopt.engine import trainer as trainer_mod

print("file:", inspect.getfile(trainer_mod))
classes = [n for n in dir(trainer_mod) if not n.startswith("_") and isinstance(getattr(trainer_mod, n), type)]
print("public classes:", classes)
print()

Trainer = trainer_mod.ReflACTTrainer  # the actual class name
print("Trainer.__init__:", inspect.signature(Trainer.__init__))
print()
print("Trainer methods:")
for m in [m for m in dir(Trainer) if not m.startswith("_") and callable(getattr(Trainer, m))]:
    print(f"  - {m}")

file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/engine/trainer.py
public classes: ['BatchSpec', 'EnvAdapter', 'ReflACTTrainer', 'defaultdict']

Trainer.__init__: (self, cfg: 'dict', adapter: 'EnvAdapter') -> 'None'

Trainer methods:
  - train


In [5]:
src = inspect.getsource(Trainer)
lines = src.splitlines()
print(f"ReflACTTrainer class — {len(lines)} lines total\n")

# Find the train method
loop_starts = [i for i, line in enumerate(lines) if "def train" in line or "def fit" in line or "def run" in line]
print(f"loop method candidates at lines: {loop_starts}")
print()

if loop_starts:
    start = loop_starts[0]
    print("=" * 70)
    print(f"From {lines[start].strip()}:")
    print("=" * 70)
    print("\n".join(lines[start : start + 60]))

ReflACTTrainer class — 1413 lines total

loop method candidates at lines: [16]

From def train(self) -> dict::
    def train(self) -> dict:
        """Execute the full ReflACT training loop. Returns summary dict."""
        cfg = self.cfg
        adapter = self.adapter
        out_root = cfg["out_root"]
        os.makedirs(out_root, exist_ok=True)

        # ── Adapter setup (one-time init) ────────────────────────────
        adapter.setup(cfg)
        dataloader = adapter.get_dataloader()

        def _build_train_env(batch: BatchSpec):
            env_manager = adapter.build_env_from_batch(batch, out_root=out_root)
            return env_manager, batch.batch_size, batch.seed

        def _build_eval_env(split: str, env_num: int, seed: int):
            if dataloader is None:
                env_manager = adapter.build_eval_env(
                    env_num=env_num,
                    split=split,
                    seed=seed,
                    out_root=out_root,
                )

In [6]:
# Find the epoch loop — look for 'for epoch in' or 'while step'
epoch_loop = [i for i, line in enumerate(lines) if "for epoch" in line or "for step" in line or "for iteration" in line]
print(f"epoch/step/iteration loops at lines: {epoch_loop}")
print()
if epoch_loop:
    start = epoch_loop[0]
    print(f"--- excerpt around line {start} ---")
    print("\n".join(lines[start : start + 40]))

epoch/step/iteration loops at lines: [379, 415, 851]

--- excerpt around line 379 ---
        for epoch in range(1, num_epochs + 1):
            if dataloader is not None:
                epoch_batches = dataloader.plan_train_epoch(
                    epoch=epoch,
                    steps_per_epoch=steps_per_epoch,
                    accumulation=accumulation,
                    batch_size=batch_size,
                    seed=seed,
                    out_root=out_root,
                )
                shuffled_seeds = [batch.seed for batch in epoch_batches]
            else:
                epoch_batches = []
                epoch_rng = random.Random(seed + epoch * 1000)
                shuffled_seeds = base_seeds.copy()
                epoch_rng.shuffle(shuffled_seeds)

            # Step buffer: accumulates per-step context (failure patterns +
            # rejected edits) within this epoch so optimizers see full history.
            step_buffer: list[dict] = []
            act

## 2. The four-step loop in modules

The "rollout → reflect → edit → gate" loop the paper describes is implemented across four packages. Look at each in turn.

In [7]:
import importlib

from skillopt import gradient

print("=== gradient/ (the reflection step) ===")
print("  file:", inspect.getfile(gradient))
print("  public:", [n for n in dir(gradient) if not n.startswith("_")][:10])
print()

print("=== optimizer/ (the edit step) ===")
opt_pkg = importlib.import_module("skillopt.optimizer")
print("  file:", inspect.getfile(opt_pkg))
print("  modules:", sorted([m for m in dir(opt_pkg) if not m.startswith("_")])[:15])
print()

print("=== evaluation/ (the gate step) ===")
from skillopt.evaluation import gate as gate_mod

print("  file:", inspect.getfile(gate_mod))
print("  public:", [n for n in dir(gate_mod) if not n.startswith("_")][:10])

=== gradient/ (the reflection step) ===
  file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/gradient/__init__.py
  public: ['aggregate', 'merge_patches', 'reflect', 'run_minibatch_reflect']

=== optimizer/ (the edit step) ===
  file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/optimizer/__init__.py
  modules: ['apply_edit', 'apply_patch', 'clip', 'lr_autonomous', 'meta_skill', 'rank_and_select', 'rewrite', 'scheduler', 'skil
l', 'slow_update', 'update_modes']

=== evaluation/ (the gate step) ===
  file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/evaluation/gate.py
  public: ['GateAction', 'GateResult', 'Literal', 'annotations', 'dataclass', 'evaluate_gate']


### The gate — `evaluate_gate(...)`

This is the smallest of the four — it decides whether a proposed edit clears the held-out validation bar.

In [8]:
print(inspect.getsource(gate_mod.evaluate_gate)[:2500])

def evaluate_gate(
    candidate_skill: str,
    cand_hard: float,
    current_skill: str,
    current_score: float,
    best_skill: str,
    best_score: float,
    best_step: int,
    global_step: int,
) -> GateResult:
    """Pure gate decision: compare candidate score to current/best.

    Returns a *GateResult* with updated state; the caller decides what
    to do with it (print, mutate trainer state, log, etc.).
    """
    if cand_hard > current_score:
        new_current_skill = candidate_skill
        new_current_score = cand_hard
        if cand_hard > best_score:
            return GateResult(
                action="accept_new_best",
                current_skill=new_current_skill,
                current_score=new_current_score,
                best_skill=candidate_skill,
                best_score=cand_hard,
                best_step=global_step,
            )
        return GateResult(
            action="accept",
            current_skill=new_current_skill,
            cu

## 3. The benchmark adapters — `envs/`

Each benchmark (SearchQA, ALFWorld, DocVQA, SpreadsheetBench, OfficeQA, LiveMath) plugs in through the same `EnvAdapter` interface. See the abstract base:

In [9]:
from skillopt.envs import base as env_base

print("file:", inspect.getfile(env_base))
print()
# Show just the class signature
src = inspect.getsource(env_base.EnvAdapter)
print(src[:2500])

file: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/envs/base.py

class EnvAdapter(ABC):
    """Abstract adapter for connecting ReflACT to any environment.

    Subclasses must implement all abstract methods. The ReflACT trainer
    calls these methods at the appropriate pipeline stages.
    """

    # ── Lifecycle hooks ────────────────────────────────────────────────────

    def setup(self, cfg: dict) -> None:
        """Called once by the trainer before the training loop begins.

        Override to perform one-time initialization that requires the full
        config (e.g., data loading, split creation).  Default is a no-op.
        """
        self._cfg = dict(cfg)

    def get_dataloader(self) -> BaseDataLoader | None:
        """Return the task dataloader used by this adapter, if any."""
        return None

    def requires_ray(self) -> bool:
        """Return whether this adapter requires Ray runtime initialization."""
        return False

    def build_reference_text(sel

## 4. Data sources

| Source | Path |
|---|---|
| SkillOpt library | `~/Documents/GitHub/SkillOpt` (cloned from `github.com/microsoft/SkillOpt`) |
| Installed copy | editable into `~/Documents/GitHub/abook/.venv` |
| Use-case framing | `~/Documents/GitHub/_docs/notebook/use-cases/02-skillopt.md` |

Every code snippet above is real `inspect.getsource(...)` output from the installed package, not transcribed prose.

→ **Next:** [`02-gradient-deep-dive.ipynb`](02-gradient-deep-dive.ipynb).

## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. The Trainer — `engine/trainer.py`
- 2. The four-step loop in modules
- 3. The benchmark adapters — `envs/`
- 4. Data sources

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/01-architecture-tour.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")